In [11]:
from recbole.quick_start import load_data_and_model, run_recbole
import torch
import pandas as pd


import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation

from recbole.model.general_recommender import NeuMF
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer, init_seed, init_logger
from collections import defaultdict
import os



In [2]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 0. LOAD MODEL AND DATASET ───────────────────────────────────────────────
from recbole.quick_start import load_data_and_model

config, model, dataset, train_data, valid_data, test_data = load_data_and_model(
    model_file='saved/sasrec_ml-1m.pth'
)
model.eval()
device = config['device']

05 May 09:49    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True


data_path = dataset/ml-1mm
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}
repeatable = True
metrics = ['Recall', 'NDCG', 'Hit', 'Deep_LT_Coverage', 'GiniIndex', 'AveragePopularity', 'ItemCoverage', 'NDCGTail', 'NDCGHead', 'NDCGMid']
topk = [10]
valid_metric = NDCG@10
valid_metric_bigger = True
eval_batch_size = 4096
metric_decimal_place = 4

Dataset Hyper Parameters:
field_separator = 	
seq_separator

In [3]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict



model.eval()
device = config['device']

# ── 1. LOAD RAW METADATA ────────────────────────────────────────────────────
# read the raw file and inspect first

movies = pd.read_csv(
    'dataset/ml-1mm/ml-1mm.item',
    sep='\t',
    engine='python'
)

# rename to simple names
movies = movies.rename(columns={
    'item_id:token':        'item_id',
    'movie_title:token_seq': 'title',
    'release_year:token':   'year',
    'genre:token_seq':      'genre'
})

# genres are space-separated in this file (not pipe-separated)
# e.g. "Animation Children's Comedy" instead of "Animation|Children|Comedy"
# so split on space when building genre pools


ratings = pd.read_csv(
    'dataset/ml-1mm/ml-1mm.inter',
    sep='\t',
    engine='python'
)


# rename to simple names
ratings = ratings.rename(columns={
    'user_id:token':    'user_id',
    'item_id:token':    'item_id',
    'rating:float':     'rating',
    'timestamp:float':  'timestamp'
})


# map raw item IDs to RecBole internal IDs
item_id_map = dataset.field2token_id['item_id']
movies['internal_id'] = movies['item_id'].astype(str).map(item_id_map)
movies = movies.dropna(subset=['internal_id'])
movies['internal_id'] = movies['internal_id'].astype(int)
movies = movies.sort_values('internal_id').reset_index(drop=True)



n_items = dataset.item_num
n_users = dataset.user_num



# ── 2. DEFINE CONCEPT ITEM POOLS ────────────────────────────────────────────
# split each genre string by space, flatten, and deduplicate
all_unique_genres = sorted(set(
    g.strip()
    for genres in movies['genre'].dropna()
    for g in str(genres).split(' ')
    if g.strip()
))


# genre pools — items belonging to each genre
genre_pools = defaultdict(list)
for _, row in movies.iterrows():
    iid = int(row['internal_id'])
    for g in str(row['genre']).split(' '):   
        g = g.strip()
        if g in all_unique_genres:                 
            genre_pools[g].append(iid )



In [4]:
# popularity pool — top 10% most interacted items
inter_df = pd.DataFrame({
    'user_id': dataset.inter_feat['user_id'].numpy(),
    'item_id': dataset.inter_feat['item_id'].numpy(),
})


item_counts = inter_df.groupby('item_id')['user_id'].count()
pop_map = {}
for raw_id, count in item_counts.items():
    #iid = item_id_map.get(str(raw_id))
    iid = int(raw_id)

    if iid is not None:
        pop_map[int(iid)] = count

counts_arr = np.array([pop_map.get(i, 0) for i in range(n_items)])
pop_threshold = np.percentile(counts_arr, 90)
popularity_pool = [i for i in range(n_items) if counts_arr[i] >= pop_threshold]

In [5]:
# niche pool — bottom 20% least interacted items
niche_threshold = np.percentile(counts_arr, 10)
niche_pool = [i for i in range(n_items) if 0 < counts_arr[i] <= niche_threshold]


In [6]:
# era pools
movies['year'] = pd.to_numeric(movies['year'], errors='coerce')

classic_pool   = [int(r['internal_id']) for _, r in movies.iterrows() if r['year'] < 1970]
retro_pool     = [int(r['internal_id']) for _, r in movies.iterrows() if 1970 <= r['year'] < 1990]
modern_pool    = [int(r['internal_id']) for _, r in movies.iterrows() if 1990 <= r['year'] < 2000]
contemporary_pool = [int(r['internal_id']) for _, r in movies.iterrows() if r['year'] >= 2000]









In [7]:
# Concepts we'll predict (in fixed order)
genre_concepts = list(all_unique_genres)              # e.g. ['Action', 'Comedy', ...]
scalar_concepts = ['popularity', 'niche',
                   'classic', 'retro', 'modern', 'contemporary']
concept_names = genre_concepts + scalar_concepts
N_CONCEPTS = len(concept_names)
print(f"Total concepts: {N_CONCEPTS}")

# Lookup: item_id -> set of genres
item_to_genres = {}
for _, row in movies.iterrows():
    item_to_genres[int(row['internal_id'])] = set(row['genre'].split('|'))




# Lookup: item_id -> year bucket
item_to_era = {}
for _, row in movies.iterrows():
    y = row['year']
    if pd.isna(y):                  era = None
    elif y < 1970:                  era = 'classic'
    elif y < 1990:                  era = 'retro'
    elif y < 2000:                  era = 'modern'
    else:                           era = 'contemporary'
    item_to_era[int(row['internal_id'])] = era



# Sets for popularity / niche
pop_set   = set(popularity_pool)
niche_set = set(niche_pool)


Total concepts: 24


In [8]:



def compute_user_concepts(item_seq):
    """item_seq: list/array of item IDs (padding 0s allowed). Returns [N_CONCEPTS] vector."""
    items = [i for i in item_seq if i != 0]
    if len(items) == 0:
        return np.zeros(N_CONCEPTS, dtype=np.float32)

    L = len(items)
    vec = np.zeros(N_CONCEPTS, dtype=np.float32)

    # genre fractions
    genre_idx = {g: i for i, g in enumerate(genre_concepts)}
    for it in items:
        for g in item_to_genres.get(it, []):
            if g in genre_idx:
                vec[genre_idx[g]] += 1.0
    vec[:len(genre_concepts)] /= L     # normalize to fractions

    # popularity / niche fractions
    vec[len(genre_concepts) + 0] = sum(1 for it in items if it in pop_set)   / L
    vec[len(genre_concepts) + 1] = sum(1 for it in items if it in niche_set) / L

    # era fractions
    era_offset = len(genre_concepts) + 2
    era_idx = {'classic': 0, 'retro': 1, 'modern': 2, 'contemporary': 3}

    era_count = 0
    for it in items:
        e = item_to_era.get(it)
        if e in era_idx:
            vec[era_offset + era_idx[e]] += 1.0
            era_count += 1
    if era_count > 0:
        vec[era_offset:era_offset+4] /= era_count

    return vec

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SASRecCBM(nn.Module):
    """
    Predictive concept bottleneck on top of frozen SASRec.

    Pipeline:  h → concept predictor → ĉ → reconstructor → ĥ → item scores
    """
    def __init__(self, sasrec, n_concepts, hidden_size=128):
        super().__init__()
        self.sasrec = sasrec                         # frozen encoder
        for p in self.sasrec.parameters():
            p.requires_grad = False

        self.hidden_size = hidden_size
        self.n_concepts  = n_concepts

        # h -> ĉ
        self.concept_predictor = nn.Sequential(
        nn.Linear(hidden_size, 256),
        nn.BatchNorm1d(256),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.1),

        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.1),

        nn.Linear(128, n_concepts),
        nn.Sigmoid(),
    )

        self.reconstructor = nn.Sequential(
        nn.Linear(n_concepts, 64),
        nn.BatchNorm1d(64),
        nn.LeakyReLU(0.1),

        nn.Linear(64, hidden_size),
)
       

       

    def encode(self, item_seq, item_seq_len):
        with torch.no_grad():
            return self.sasrec.forward(item_seq, item_seq_len)   # [B, 128]

    def forward(self, item_seq, item_seq_len):
        h     = self.encode(item_seq, item_seq_len)              # [B, 128]
        c_hat = self.concept_predictor(h)                        # [B, N_CONCEPTS]
        h_hat = self.reconstructor(c_hat)                        # [B, 128]
        return h, c_hat, h_hat

    def score_items(self, h_hat):
        # use SASRec's own item embedding table as the prediction head
        item_emb = self.sasrec.item_embedding.weight             # [n_items, 128]
        return h_hat @ item_emb.T                                # [B, n_items]

# SASRec-only eval function

In [10]:
@torch.no_grad()
def evaluate_sasrec(sasrec, eval_data, k_values=(5, 10, 20)):
    """
    Evaluate frozen SASRec directly (no bottleneck).
    Uses the same masking/ranking logic as evaluate_cbm for a fair comparison.
    """
    sasrec.eval()

    hits  = {k: 0   for k in k_values}
    ndcgs = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n_users = 0

    for batch in eval_data:
        if isinstance(batch, tuple):
            interaction = batch[0]
        else:
            interaction = batch
        interaction  = interaction.to(device)

        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        # Forward through SASRec directly, score against item embeddings
        h        = sasrec.forward(item_seq, item_seq_len)            # [B, 128]
        item_emb = sasrec.item_embedding.weight                      # [n_items, 128]
        logits   = h @ item_emb.T                                    # [B, n_items]

        # Same masking as evaluate_cbm
        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1

        for k in k_values:
            in_top_k  = (rank <= k)
            hits[k]  += in_top_k.sum().item()
            ndcgs[k] += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()

        mrr_sum += (1.0 / rank.float()).sum().item()
        n_users += target_item.size(0)

    return {
        'hr':   {k: hits[k]  / n_users for k in k_values},
        'ndcg': {k: ndcgs[k] / n_users for k in k_values},
        'mrr':  mrr_sum / n_users,
    }

## Training and Evalaution for concept predictor

In [12]:
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import pearsonr

device = next(model.parameters()).device

cbm = SASRecCBM(model, n_concepts=N_CONCEPTS, hidden_size=model.hidden_size).to(device)
opt = torch.optim.Adam(
    [p for p in cbm.parameters() if p.requires_grad], lr=1e-2
)

LAMBDA_CONCEPT = 1.5

LAMBDA_RECON   = 0.5

N_EPOCHS       = 50
K_VALUES       = [5, 10, 20]

TOP_K_CONCEPTS = 3


# ── EVAL FUNCTION ─────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_cbm(cbm, eval_data, k_values=(5, 10, 20), top_k_concepts=3):
    cbm.eval()

    total_rec, total_con, n_batches = 0., 0., 0
    hits  = {k: 0   for k in k_values}
    ndcgs = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n_users = 0

    all_pred, all_true = [], []
    topk_true_all, topk_pred_all, topk_idx_all = [], [], []
    topk_exact_correct   = 0.0
    topk_partial_correct = 0.0

    for batch in eval_data:
        if isinstance(batch, tuple):
            interaction = batch[0]
        else:
            interaction = batch
        interaction  = interaction.to(device)

        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        gt_concepts = torch.stack([
            torch.from_numpy(compute_user_concepts(seq.cpu().tolist()))
            for seq in item_seq
        ]).to(device)

        _, c_hat, h_hat = cbm(item_seq, item_seq_len)
        logits          = cbm.score_items(h_hat)

        loss_rec = F.cross_entropy(logits, target_item)
        loss_con = F.binary_cross_entropy(c_hat, gt_concepts)
        total_rec += loss_rec.item()
        total_con += loss_con.item()
        n_batches += 1

        # ── top-k concept value capture ───────────────────────────────────────
        true_topk_vals, true_topk_idx = gt_concepts.topk(top_k_concepts, dim=1)
        pred_on_topk                  = c_hat.gather(1, true_topk_idx)

        topk_true_all.append(true_topk_vals.cpu().numpy())
        topk_pred_all.append(pred_on_topk.cpu().numpy())
        topk_idx_all.append(true_topk_idx.cpu().numpy())


        # ── strict and partial top-k set-match accuracy ──────────────────────
        true_sorted = true_topk_idx.sort(dim=1).values
        pred_sorted = c_hat.topk(top_k_concepts, dim=1).indices.sort(dim=1).values

        exact_match = (pred_sorted == true_sorted).all(dim=1).float()
        topk_exact_correct += exact_match.sum().item()

        for u in range(true_sorted.size(0)):
            ov = len(set(pred_sorted[u].tolist()) & set(true_sorted[u].tolist()))
            topk_partial_correct += ov / top_k_concepts

        all_pred.append(c_hat.cpu().numpy())
        all_true.append(gt_concepts.cpu().numpy())

        # ── recommendation metrics ────────────────────────────────────────────
        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1

        for k in k_values:
            in_top_k  = (rank <= k)
            hits[k]  += in_top_k.sum().item()
            ndcgs[k] += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()
        mrr_sum += (1.0 / rank.float()).sum().item()
        n_users += target_item.size(0)

    # ── aggregate ─────────────────────────────────────────────────────────────
    pred = np.vstack(all_pred)
    true = np.vstack(all_true)
    topk_true = np.vstack(topk_true_all)
    topk_pred = np.vstack(topk_pred_all)
    topk_idx  = np.vstack(topk_idx_all)

    overall_mae   = float(np.mean(np.abs(pred - true)))
    baseline_mae  = float(np.mean(np.abs(true - true.mean(axis=0, keepdims=True))))
    topk_mae      = float(np.mean(np.abs(topk_pred - topk_true)))
    topk_recovery = float(np.mean(
        np.clip(topk_pred / np.clip(topk_true, 1e-6, None), 0, 2)
    ))
    topk_corr     = float(pearsonr(topk_true.flatten(), topk_pred.flatten()).statistic)

    return {
        'rec_loss':         total_rec / n_batches,
        'con_loss':         total_con / n_batches,
        'concept_mae':      overall_mae,
        'baseline_mae':     baseline_mae,
        'topk_mae':         topk_mae,
        'topk_recovery':    topk_recovery,
        'topk_corr':        topk_corr,
        'topk_exact_acc':   topk_exact_correct   / n_users,
        'topk_partial_acc': topk_partial_correct / n_users,
        'topk_true':        topk_true,
        'topk_pred':        topk_pred,
        'topk_idx':         topk_idx,
        'hr':               {k: hits[k]  / n_users for k in k_values},
        'ndcg':             {k: ndcgs[k] / n_users for k in k_values},
        'mrr':              mrr_sum / n_users,
    }
## Evaluate the frozen SASRec as a baseline before training the CBM

print("Evaluating frozen SASRec baseline...")
sasrec_baseline = evaluate_sasrec(model, test_data, k_values=K_VALUES)
print(f"  SASRec  HR@10={sasrec_baseline['hr'][10]:.4f}  "
      f"NDCG@10={sasrec_baseline['ndcg'][10]:.4f}  "
      f"MRR={sasrec_baseline['mrr']:.4f}\n")


BEST_METRIC = 'hr'   ## metrics name
BEST_K      = 10     # for hr/ndcg
SAVE_PATH   = 'best_cbm_ml-1m_SASREC.pt'

best_score = -float('inf')   # use float('inf') if tracking a loss/MAE (lower is better)


# ── TRAINING LOOP ─────────────────────────────────────────────────────────────
history = []

for epoch in range(N_EPOCHS):
    cbm.train()
    total_rec, total_con, n_batches = 0., 0., 0

    for batch in train_data:
        batch        = batch.to(device)
        item_seq     = batch['item_id_list']
        item_seq_len = batch['item_length']
        target_item  = batch['item_id']
        
        #print(f'item_seq.shape: {item_seq.shape}')
        gt_concepts = torch.stack([
            torch.from_numpy(compute_user_concepts(seq.cpu().tolist()))
            for seq in item_seq
        ]).to(device)
       
        h, c_hat, h_hat = cbm(item_seq, item_seq_len)
        logits          = cbm.score_items(h_hat)

        loss_rec = F.cross_entropy(logits, target_item)

        loss_recon = F.mse_loss(h_hat, h.detach())


        loss_con = F.binary_cross_entropy(c_hat, gt_concepts)
        #loss     = loss_rec + LAMBDA_CONCEPT * loss_con

        loss = loss_rec + LAMBDA_CONCEPT * loss_con + LAMBDA_RECON * loss_recon


        opt.zero_grad()
        loss.backward()
        opt.step()

        total_rec += loss_rec.item()
        total_con += loss_con.item()
        n_batches += 1

    train_rec = total_rec / n_batches
    train_con = total_con / n_batches

    # Evaluate on test set after this epoch
    test = evaluate_cbm(cbm, test_data,
                        k_values=K_VALUES, top_k_concepts=TOP_K_CONCEPTS)
    
    # Inside the loop, after `test = evaluate_cbm(...)`:
    current_score = test['hr'][BEST_K]   # or test['ndcg'][BEST_K], test['mrr'], etc.

    if current_score > best_score:
        best_score = current_score
        torch.save({
        'epoch': epoch + 1,
        'model_state_dict': cbm.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'metrics': test,
        'n_concepts': N_CONCEPTS,
        'hidden_size': model.hidden_size,
        }, SAVE_PATH)
        print(f"  ↳ Saved best model (HR@{BEST_K}={current_score:.4f})")
    history.append({'epoch':     epoch + 1,
                    'train_rec': train_rec,
                    'train_con': train_con,
                    'rec_loss':         test['rec_loss'],
                    'con_loss':         test['con_loss'],
                    'concept_mae':      test['concept_mae'],
                    'baseline_mae':     test['baseline_mae'],
                    'topk_mae':         test['topk_mae'],
                    'topk_recovery':    test['topk_recovery'],
                    'topk_corr':        test['topk_corr'],
                    'hr':               test['hr'],
                    'ndcg':             test['ndcg'],
                    'mrr':              test['mrr']})

    print(
    f"Epoch {epoch+1:2d} | "
    f"train: rec={train_rec:.4f} con={train_con:.4f} | "
    f"test: rec={test['rec_loss']:.4f} con={test['con_loss']:.4f} | "
    f"MAE={test['concept_mae']:.4f} (base={test['baseline_mae']:.4f}) | "
    f"top{TOP_K_CONCEPTS}_acc={test['topk_exact_acc']:.3f} "
    f"(partial={test['topk_partial_acc']:.3f}) | "
    f"HR@10={test['hr'][10]:.4f} NDCG@10={test['ndcg'][10]:.4f}"
)




Evaluating frozen SASRec baseline...
  SASRec  HR@10=0.2969  NDCG@10=0.1714  MRR=0.1480

  ↳ Saved best model (HR@10=0.2339)
Epoch  1 | train: rec=5.7141 con=0.1706 | test: rec=6.2853 con=0.1642 | MAE=0.0498 (base=0.0425) | top3_acc=0.412 (partial=0.786) | HR@10=0.2339 NDCG@10=0.1247
  ↳ Saved best model (HR@10=0.2599)
Epoch  2 | train: rec=5.5421 con=0.1561 | test: rec=6.1942 con=0.1596 | MAE=0.0453 (base=0.0425) | top3_acc=0.435 (partial=0.794) | HR@10=0.2599 NDCG@10=0.1419
Epoch  3 | train: rec=5.5089 con=0.1537 | test: rec=6.2493 con=0.1591 | MAE=0.0450 (base=0.0425) | top3_acc=0.446 (partial=0.798) | HR@10=0.2578 NDCG@10=0.1451
  ↳ Saved best model (HR@10=0.2611)
Epoch  4 | train: rec=5.4908 con=0.1523 | test: rec=6.2337 con=0.1572 | MAE=0.0428 (base=0.0425) | top3_acc=0.454 (partial=0.801) | HR@10=0.2611 NDCG@10=0.1423
  ↳ Saved best model (HR@10=0.2644)
Epoch  5 | train: rec=5.4796 con=0.1514 | test: rec=6.2074 con=0.1567 | MAE=0.0425 (base=0.0425) | top3_acc=0.440 (partial=0.79

In [24]:
pd.DataFrame(history).to_csv('cbm_training_history_SASREC_ml-1m.csv', index=False)

## Loading CBM model

In [25]:
# Rebuild the architecture first (must match what you trained)
cbm = SASRecCBM(model, n_concepts=N_CONCEPTS, hidden_size=model.hidden_size).to(device)

# Load the checkpoint
checkpoint = torch.load(SAVE_PATH, map_location=device)
cbm.load_state_dict(checkpoint['model_state_dict'])
cbm.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Best metrics: HR@10={checkpoint['metrics']['hr'][10]:.4f}")

/tmp/ipykernel_374736/1170604903.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(SAVE_PATH, map_location=device)


Loaded model from epoch 49
Best metrics: HR@10=0.2916


## Steering

In [63]:
import numpy as np
import torch
import torch.nn.functional as F


# ── CONFIGURATION ─────────────────────────────────────────────────────────────
POP_CONCEPT_IDX = 18         # <-- set this to your popularity concept's index
SCALE_FACTOR    = -0.1    # how much to suppress (0.0 = fully off, 1.0 = unchanged)
K_FOR_TOPK      = 20         # top-K recommendations for coverage / Gini


# ── STEERING-AWARE RECOMMENDATION ─────────────────────────────────────────────
@torch.no_grad()
def get_topk_recommendations(cbm, eval_data, k=10,
                             concept_idx=None, scale=1.0):
    """
    Generate top-k item recommendations for every user in eval_data.
    If concept_idx is given, multiply that concept's activation by `scale`
    before reconstructing h_hat and scoring items.
    Returns a flat numpy array of recommended item ids (length = n_users * k).
    """
    cbm.eval()
    all_recs = []

    for batch in eval_data:
        interaction  = batch[0] if isinstance(batch, tuple) else batch
        interaction  = interaction.to(device)
        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        print(f'item_seq.shape: {item_seq.shape}')
        # Forward pass through CBM (we ignore h_hat here because we re-derive it)
        h, c_hat, _ = cbm(item_seq, item_seq_len)

        # ── STEERING: scale the chosen concept's activation ───────────────────
        if concept_idx is not None:
            c_hat = c_hat.clone()
            print(f'c_hat_before {c_hat[:, concept_idx].mean()}')
            c_hat[:, concept_idx] = c_hat[:, concept_idx] * scale
            print(f'c_hat_after {c_hat[:, concept_idx].mean()}')

        # Re-derive h_hat from the (possibly steered) concept activations.
        # This assumes cbm has a c_to_h decoder layer; adjust the attribute
        # name to match your implementation if it differs.
        h_hat   = cbm.reconstructor(c_hat)        # <-- adjust if needed
        logits  = cbm.score_items(h_hat)

        # Mask padding token and items already in the user's history
        logits[:, 0] = -float('inf')
        logits.scatter_(1, item_seq, -float('inf'))

        topk_items = logits.topk(k, dim=1).indices  # [batch, k]
        all_recs.append(topk_items.cpu().numpy())

    return np.concatenate(all_recs, axis=0)  # [n_users, k]


# ── COVERAGE + GINI ───────────────────────────────────────────────────────────
def coverage(recs, n_items):
    """Fraction of the catalog that appears at least once in any user's top-k."""
    unique_recommended = np.unique(recs)
    # exclude padding id 0 if present
    unique_recommended = unique_recommended[unique_recommended != 0]
    return len(unique_recommended) / (n_items - 1)   # -1 for the padding slot


def gini(recs, n_items):
    """
    Gini coefficient of recommendation frequency across the catalog.
    0 = perfectly uniform exposure, 1 = all recs concentrated on one item.
    """
    counts = np.bincount(recs.flatten(), minlength=n_items).astype(np.float64)
    counts = counts[1:]                       # drop padding slot
    counts = np.sort(counts)
    n      = len(counts)
    if counts.sum() == 0:
        return 0.0
    cum = np.cumsum(counts)
    # standard Gini formula
    return (2 * np.sum((np.arange(1, n + 1)) * counts) - (n + 1) * cum[-1]) \
           / (n * cum[-1])


# ── RUN THE EXPERIMENT ────────────────────────────────────────────────────────
n_items = cbm.n_items if hasattr(cbm, 'n_items') else model.n_items

print(f"Generating baseline recommendations (no steering)...")
recs_base = get_topk_recommendations(cbm, test_data, k=K_FOR_TOPK)

print(f"Generating steered recommendations (popularity × {SCALE_FACTOR})...")
recs_steer = get_topk_recommendations(
    cbm, test_data, k=K_FOR_TOPK,
    concept_idx=POP_CONCEPT_IDX, scale=SCALE_FACTOR,
)

cov_base,  gini_base  = coverage(recs_base,  n_items), gini(recs_base,  n_items)
cov_steer, gini_steer = coverage(recs_steer, n_items), gini(recs_steer, n_items)

print("\n── RESULTS ─────────────────────────────────")
print(f"Baseline:  Coverage = {cov_base :.4f}   Gini = {gini_base :.4f}")
print(f"Steered:   Coverage = {cov_steer:.4f}   Gini = {gini_steer:.4f}")


Generating baseline recommendations (no steering)...
item_seq.shape: torch.Size([4096, 50])
item_seq.shape: torch.Size([1944, 50])
Generating steered recommendations (popularity × -0.1)...
item_seq.shape: torch.Size([4096, 50])
c_hat_before 0.42729535698890686
c_hat_after -0.04272953420877457
item_seq.shape: torch.Size([1944, 50])
c_hat_before 0.427034854888916
c_hat_after -0.04270348697900772

── RESULTS ─────────────────────────────────
Baseline:  Coverage = 0.7699   Gini = 0.6835
Steered:   Coverage = 0.6074   Gini = 0.8097


In [67]:
import numpy as np


def popularity_exposure(recs, popularity_pool):
    """
    Fraction of recommended items that are in the popularity pool.
    `recs` has shape [n_users, k].
    Returns:
        overall_rate:   float, share of all top-k slots that are popular
        per_user_rate:  np.array of shape [n_users]
    """
    is_pop = np.isin(recs, list(popularity_pool))
    per_user_rate = is_pop.mean(axis=1)
    overall_rate  = is_pop.mean()
    return overall_rate, per_user_rate


# ── RUN ───────────────────────────────────────────────────────────────────────
print(f"Generating baseline recommendations (no steering)...")
recs_base = get_topk_recommendations(cbm, test_data, k=K_FOR_TOPK)

print(f"Generating steered recommendations (popularity × {SCALE_FACTOR})...")
recs_steer = get_topk_recommendations(
    cbm, test_data, k=K_FOR_TOPK,
    concept_idx=POP_CONCEPT_IDX, scale=SCALE_FACTOR,
)

pop_base,  per_user_base  = popularity_exposure(recs_base,  popularity_pool)
pop_steer, per_user_steer = popularity_exposure(recs_steer, popularity_pool)

print("\n── RESULTS ─────────────────────────────────")
print(f"Baseline:  Popularity exposure = {pop_base :.4f}  "
      f"(median per user = {np.median(per_user_base ):.4f})")
print(f"Steered:   Popularity exposure = {pop_steer:.4f}  "
      f"(median per user = {np.median(per_user_steer):.4f})")
print(f"Δ Exposure = {pop_steer - pop_base:+.4f}  "
      f"(negative = successful suppression)")

Generating baseline recommendations (no steering)...
item_seq.shape: torch.Size([4096, 50])
item_seq.shape: torch.Size([1944, 50])
Generating steered recommendations (popularity × -0.1)...
item_seq.shape: torch.Size([4096, 50])
c_hat_before 0.42729535698890686
c_hat_after -0.04272953420877457
item_seq.shape: torch.Size([1944, 50])
c_hat_before 0.427034854888916
c_hat_after -0.04270348697900772

── RESULTS ─────────────────────────────────
Baseline:  Popularity exposure = 0.4108  (median per user = 0.4000)
Steered:   Popularity exposure = 0.1039  (median per user = 0.0000)
Δ Exposure = -0.3068  (negative = successful suppression)


In [83]:
import torch
import torch.nn.functional as F
import numpy as np


# ── CONFIGURATION ─────────────────────────────────────────────────────────────
POP_CONCEPT_IDX = 18         # <-- set this to your popularity concept's index
SCALE_FACTOR    = 0.85    # how much to suppress (0.0 = fully off, 1.0 = unchanged)
K_FOR_TOPK      = 20         # top-K recommendations for coverage / Gini


@torch.no_grad()
def evaluate_with_steering(cbm, eval_data, popularity_pool, k=10,
                           concept_idx=None, scale=1.0):
    """
    Compute HR@k, NDCG@k, and popularity exposure, with optional steering.
    Set concept_idx=None for the unsteered baseline.
    """
    cbm.eval()

    hits     = 0
    ndcg_sum = 0.0
    n_users  = 0

    pop_pool_arr = np.array(list(popularity_pool))
    pop_count    = 0
    total_recs   = 0

    for batch in eval_data:
        interaction  = batch[0] if isinstance(batch, tuple) else batch
        interaction  = interaction.to(device)
        item_seq     = interaction['item_id_list']
        item_seq_len = interaction['item_length']
        target_item  = interaction['item_id']

        h, c_hat, _ = cbm(item_seq, item_seq_len)

        # ── Apply steering ────────────────────────────────────────────────────
        if concept_idx is not None:
            c_hat = c_hat.clone()
            c_hat[:, concept_idx] = c_hat[:, concept_idx] * scale

        h_hat  = cbm.reconstructor(c_hat)
        logits = cbm.score_items(h_hat)

        # Mask padding and user history
        scores = logits.clone()
        scores[:, 0] = -float('inf')
        scores.scatter_(1, item_seq, -float('inf'))

        # ── HR@k and NDCG@k ───────────────────────────────────────────────────
        target_scores = scores.gather(1, target_item.view(-1, 1))
        rank          = (scores > target_scores).sum(dim=1) + 1

        in_top_k  = (rank <= k)
        hits     += in_top_k.sum().item()
        ndcg_sum += (in_top_k.float() / torch.log2(rank.float() + 1)).sum().item()
        n_users  += target_item.size(0)

        # ── Popularity exposure ───────────────────────────────────────────────
        topk_items = scores.topk(k, dim=1).indices.cpu().numpy()
        pop_count   += np.isin(topk_items, pop_pool_arr).sum()
        total_recs  += topk_items.size

    return {
        f'hr@{k}':         hits / n_users,
        f'ndcg@{k}':       ndcg_sum / n_users,
        'pop_exposure':    pop_count / total_recs,
    }


# ── RUN ───────────────────────────────────────────────────────────────────────
K = 10

print("Evaluating baseline (no steering)...")
base = evaluate_with_steering(cbm, test_data, popularity_pool, k=K)

print(f"Evaluating steered (popularity × {SCALE_FACTOR})...")
steer = evaluate_with_steering(cbm, test_data, popularity_pool, k=K,
                               concept_idx=POP_CONCEPT_IDX, scale=SCALE_FACTOR)

print("\n── RESULTS ─────────────────────────────────")
print(f"{'Metric':<18} {'Baseline':>10} {'Steered':>10} {'Δ':>10}")
print("─" * 52)
for key in [f'hr@{K}', f'ndcg@{K}', 'pop_exposure']:
    b, s = base[key], steer[key]
    print(f"{key:<18} {b:>10.4f} {s:>10.4f} {s-b:>+10.4f}")

Evaluating baseline (no steering)...
Evaluating steered (popularity × 0.85)...

── RESULTS ─────────────────────────────────
Metric               Baseline    Steered          Δ
────────────────────────────────────────────────────
hr@10                  0.2916     0.2833    -0.0083
ndcg@10                0.1633     0.1594    -0.0040
pop_exposure           0.4461     0.4053    -0.0408
